
# Mouse path visualization

Load CAPTCHA attempt recordings from `website/attempts/sessions.ndjson` and plot mouse paths.
Use the selection cell to explore a single captcha stage/attempt, or run the last cell to
plot all `captcha-1` attempts for every session.


In [ ]:

from pathlib import Path
import json
import matplotlib.pyplot as plt

ATTEMPTS_PATH = Path("website/attempts/sessions.ndjson")


In [ ]:

# Load each ndjson line into a session dict
sessions = []
with ATTEMPTS_PATH.open() as fh:
    for line in fh:
        if line.strip():
            sessions.append(json.loads(line))

print(f"Loaded {len(sessions)} session(s)")
print(f"Session ids: {[s.get('session_id') for s in sessions]}")


In [ ]:

def get_mouse_path(session, captcha_key="captcha-0", attempt_number=None):
    '''Return a list of {x, y, t} dicts for the selected captcha stage.

    For captcha flows that store multiple attempts (e.g. captcha-1),
    pass `attempt_number` to pull from attempt_metadata.
    '''
    captcha = session.get(captcha_key) or {}

    if isinstance(captcha.get("mouse_path"), list) and captcha["mouse_path"]:
        path = captcha["mouse_path"]
    elif attempt_number is not None:
        attempt_key = str(attempt_number)
        attempt = (captcha.get("attempt_metadata") or {}).get(attempt_key, {})
        path = attempt.get("mouse_path", [])
    else:
        path = []

    return [
        {"x": float(p["x"]), "y": float(p["y"]), "t": int(p["t"])}
        for p in path
        if isinstance(p, dict) and {"x", "y", "t"} <= p.keys()
    ]


In [ ]:

def plot_mouse_path(mouse_path, title):
    # Plot a mouse path with a position-over-time companion plot
    if not mouse_path:
        raise ValueError("No mouse_path data provided")

    x = [p["x"] for p in mouse_path]
    y = [p["y"] for p in mouse_path]
    timestamps = [p["t"] for p in mouse_path]

    start_time = timestamps[0]
    relative_t = [t - start_time for t in timestamps]

    fig, (ax_path, ax_series) = plt.subplots(1, 2, figsize=(12, 5))

    sc = ax_path.scatter(x, y, c=relative_t, cmap="viridis", s=12, label="mouse")
    ax_path.plot(x, y, color="gray", alpha=0.3)
    ax_path.invert_yaxis()  # screen space origin is typically top-left
    ax_path.set_xlabel("x")
    ax_path.set_ylabel("y")
    ax_path.set_title(title)
    fig.colorbar(sc, ax=ax_path, label="ms since start")

    ax_series.plot(relative_t, x, label="x")
    ax_series.plot(relative_t, y, label="y")
    ax_series.set_xlabel("ms since start")
    ax_series.set_title("Position over time")
    ax_series.legend()

    fig.tight_layout()
    plt.show()


In [ ]:

# Choose which session/captcha/attempt to visualize
SESSION_INDEX = 0            # which row from sessions list
CAPTCHA_KEY = "captcha-0"   # e.g. "captcha-0" or "captcha-1"
ATTEMPT_NUMBER = 1           # only used when the captcha stores attempts

session = sessions[SESSION_INDEX]
mouse_path = get_mouse_path(session, CAPTCHA_KEY, ATTEMPT_NUMBER)
print(f"Mouse path points: {len(mouse_path)}")
print("First 3 points:")
for point in mouse_path[:3]:
    print(point)


In [ ]:

# Plot the selected mouse path
plot_mouse_path(
    mouse_path,
    title=f"{CAPTCHA_KEY} (session {session.get('session_id')}, attempt {ATTEMPT_NUMBER})",
)


In [ ]:

# Plot captcha-1 mouse paths for every session and attempt
for session_index, session in enumerate(sessions):
    captcha = session.get("captcha-1") or {}
    attempt_meta = captcha.get("attempt_metadata") or {}
    if not attempt_meta:
        print(f"Session {session_index} ({session.get('session_id')}): no captcha-1 attempts")
        continue

    for attempt_key in sorted(attempt_meta, key=lambda k: int(k)):
        mouse_path = get_mouse_path(session, "captcha-1", attempt_number=int(attempt_key))
        if not mouse_path:
            print(f"Session {session_index} ({session.get('session_id')}), attempt {attempt_key}: no path")
            continue

        print(f"Plotting session {session_index} ({session.get('session_id')}), captcha-1 attempt {attempt_key}")
        plot_mouse_path(
            mouse_path,
            title=f"captcha-1 (session {session.get('session_id')}, attempt {attempt_key})",
        )
